In [1]:
# import necessary libraries
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
from sklearn.neighbors import NearestNeighbors
from sklearn.neighbors import NearestNeighbors

**For questions 1 to 3:**

Perform a linear regression to predict Y from X1, X2, and X3. Use the file homework_1.1.csv.

## Question 1

Which of the following is closest to the coefficient of X1? 

- A: 2

- B: 1

- C: 3

In [2]:
# load the data
df1= pd.read_csv('./data/homework_1.1.csv')
df1.head()

,X1,X2,X3,Y
0,-0.440646,-0.390227,0.156718,-0.877671
1,-3.810099,-1.304665,-1.105117,-10.130388
2,-1.425451,-0.340049,1.115908,0.284068
3,-1.325750,0.161906,-0.254670,-1.994344
4,3.120263,1.487343,-1.164839,2.030030


In [3]:
# fit a linear regression model
model = LinearRegression()
model.fit(df1[['X1', 'X2', 'X3']], df1['Y'])
model.coef_, model.intercept_

(array([1.00713766, 1.96456859, 2.97548854]),
 np.float64(0.0026430033444732604))

Answer : B
___

## Question 2

Which Xi has the greatest difference between the amount Y increases for each 1 unit of Xi (fixing the other Xi’s), as opposed to the amount that Y increases for each 1 unit of Xi in the dataset, on average (not fixing the other Xis)?  
**Hint: for the former, you'll have to regress Y on Xi alone, while for the latter, you'll have to regress Y on all three Xis.**

- A: X1

- B: X3

- C: X2

In [4]:
# analyze the impact of each variable
results = {}
X = df1.drop(columns=['Y'])
y = df1['Y']

for col in X.columns:
    col_res = []
    X_i = df1[[col]]

    model.fit(X_i, y)
    coeff = model.coef_[0]

    model.fit(X, y)
    coeff_full = model.coef_[X.columns.get_loc(col)]

    col_res.append(coeff)
    col_res.append(coeff_full)
    col_res.append(abs(coeff - coeff_full))

    results[col] = col_res 

pd.DataFrame(results, index=['Coefficient (Single Variable)', 'Coefficient (Full Model)', 'Difference'])

,X1,X2,X3
Coefficient (Single Variable),1.841761,4.083613,3.097041
Coefficient (Full Model),1.007138,1.964569,2.975489
Difference,0.834623,2.119044,0.121553


Answer : C
___

## Question 3
When regressing Y on all Xis together, which coefficient is most significant, considering the t-statistic as a measure of significance? 

- A: X1 

- B: X3

- C: X2 


In [5]:
X  = df1[['X1', 'X2', 'X3']]
Y = df1['Y']
X = sm.add_constant(X)  # add a constant term for the intercept

model = sm.OLS(Y, X).fit()
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      Y   R-squared:                       0.991
Model:                            OLS   Adj. R-squared:                  0.991
Method:                 Least Squares   F-statistic:                 3.543e+04
Date:                Thu, 03 Sep 2026   Prob (F-statistic):               0.00
Time:                        18:09:56   Log-Likelihood:                -727.62
No. Observations:                1000   AIC:                             1463.
Df Residuals:                     996   BIC:                             1483.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0026      0.016      0.166      0.868      -0.029       0.034
X1             1.0071      0.017     60.984      0.000       0.975       1.040
X2             1.9646      0.037     53.283      0.000       1.892       2.037
X3             2.9755      0.015    196.645      0.000       2.946       3.005
==============================================================================
Omnibus:                        0.655   Durbin-Watson:                   1.960
Prob(Omnibus):                  0.721   Jarque-Bera (JB):                0.592
Skew:                          -0.058   Prob(JB):                        0.744
Kurtosis:                       3.029   Cond. No.                         5.94
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

Answer : B
___

**For Question 4 and 5:**

Use NearestNeighbors to match data based on variables Z, given the file homework_1.2.csv.  
Pick the best match in X = 0 corresponding to each X = 1.  
Use the Z values to perform the match: a good match with X = 1 is the item whose Z value is closest to the given sample's Z value with X = 0. I suggest using sklearn's NearestNeighbors to do this, but there are many ways to do it.  

## Question 4

What is the distance of the farthest match in this set? 

- A: 0.210217  

- B: 0.0023  

- C: 0.07905

In [6]:
df2 = pd.read_csv('./data/homework_1.2.csv')
df2.head()

,X,Y,Z
0,0,0.548814,0.548814
1,1,1.215189,0.715189
2,0,0.602763,0.602763
3,0,0.544883,0.544883
4,0,0.423655,0.423655


In [7]:
# perform nearest neighbor matching
treatment = df2[df2['X'] == 1]
control = df2[df2['X'] == 0]
nn = NearestNeighbors(n_neighbors=1)
nn.fit(control[['Z']])
distances, indices = nn.kneighbors(treatment[['Z']])

In [8]:
distances.max()

np.float64(0.2102170871093757)

Answer : A
___

# Question 5  

What is the effect? (The difference between the average Y value for X = 0 values vs. the average Y value for X = 1, where the X = 0 sample has the best match for each X = 1 value).  
So we use the matched sample of X = 0 and the full sample of X = 1.

- A: 0.1689

- B: 0.5873

- C: 0.54336

In [9]:
matched_control = control.iloc[indices.flatten()]

mean_control = matched_control['Y'].mean()
mean_treatment = treatment['Y'].mean()

effect = mean_treatment - mean_control
effect

np.float64(0.5433600652185839)

Answer : C
___

**For questions 6 and 7:**

Use NearestNeighbors to match data based on variables Z, given the file homework_1.2.csv.   
Try approach B: Pick all of the matches in X = 0 that are within a distance 0.2 of each X = 1.  
Duplicates are okay, in case a given sample with X = 0 is a good match for multiple items with X = 1. 

## Question 6

How many duplicates do you end up with? (Count all but the first duplicate in each group. One way to do this is to use radius_neighbors.)

- A: 102

- B: 685

- C: 502

In [10]:
nn2 = NearestNeighbors()
nn2.fit(control[['Z']])

distances_radius, indices_radius = nn2.radius_neighbors(treatment[['Z']], radius=0.2 )

In [11]:
all_matches = np.concatenate(indices_radius)

duplicates = len(all_matches) - len(np.unique(all_matches))

duplicates

685

Answer : B
___

## Question 7

What is the effect? (Note: to compute the effect, you should take the mean of the Y values in each neighbor group, then average the Y for each group.)  

- A: 0.6654

- B: 0.1238

- C: 0.5844

In [12]:
mean_match = [control.iloc[i]["Y"].mean() for i in indices_radius]

mean_control2 = np.nanmean(mean_match)
mean_treat2 = treatment["Y"].mean()

effect = mean_treat2 - mean_control2

In [13]:
effect

np.float64(0.5844124774246182)

Answer : C
___

**For questions 1 to 3:**

Perform a linear regression to predict Y from X1, X2, and X3. Use the file homework_1.1.csv. 

## Question 1

Which of the following is closest to the coefficient of X1? 

- A: 2

- B: 1

- C: 3   


## Question 2

Which Xi has the greatest difference between the amount Y increases for each 1 unit of Xi (fixing the other Xi’s), as opposed to the amount that Y increases for each 1 unit of Xi in the dataset, on average (not fixing the other Xis)?  
**Hint: for the former, you'll have to regress Y on Xi alone, while for the latter, you'll have to regress Y on all three Xis.**

- A: X1

- B: X3

- C: X2 

## Question 3
When regressing Y on all Xis together, which coefficient is most significant, considering the t-statistic as a measure of significance? 

- A: X1 

- B: X3

- C: X2    

**For Question 4 and 5:**

Use NearestNeighbors to match data based on variables Z, given the file homework_1.2.csv.  
Pick the best match in X = 0 corresponding to each X = 1.  
Use the Z values to perform the match: a good match with X = 1 is the item whose Z value is closest to the given sample's Z value with X = 0. I suggest using sklearn's NearestNeighbors to do this, but there are many ways to do it.       

## Question 4

What is the distance of the farthest match in this set? 

- A: 0.210217  

- B: 0.0023  

- C: 0.07905   

# Question 5  

What is the effect? (The difference between the average Y value for X = 0 values vs. the average Y value for X = 1, where the X = 0 sample has the best match for each X = 1 value).  
So we use the matched sample of X = 0 and the full sample of X = 1.

- A: 0.1689

- B: 0.5873

- C: 0.54336   

**For questions 6 and 7:**

Use NearestNeighbors to match data based on variables Z, given the file homework_1.2.csv.   
Try approach B: Pick all of the matches in X = 0 that are within a distance 0.2 of each X = 1.  
Duplicates are okay, in case a given sample with X = 0 is a good match for multiple items with X = 1.     

## Question 6

How many duplicates do you end up with? (Count all but the first duplicate in each group. One way to do this is to use radius_neighbors.)

- A: 102

- B: 685

- C: 502    

## Question 7

What is the effect? (Note: to compute the effect, you should take the mean of the Y values in each neighbor group, then average the Y for each group.)  

- A: 0.6654

- B: 0.1238

- C: 0.5844